# Step 2: Warped Coordinate Prediction

Method Overview: 
0. Make sure to run assign_anchors.ipynb first to retreive the anchor image assignments first!
1. Estimates an anchor-to-test geometric transform and generates the coordinates

Be sure to specify if you would like to run evaluation
- Train
    - Find some transform between test and anchor image
    - Generate coordinates for each set of images
- Test
    - Apply the same transformation between test and DETERMINED anchor image from previous function
    - Generate coordinates for each set of images
    - EVALUATE: again the ground truth coordinates

Outputs created:
- correspondence/grouping.csv
- warped_coordinates/test_XX_warped_coordinates.csv (100 rows, x/y, 2 decimals)


In [26]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import cv2
import numpy as np
import pandas as pd

from registration_utils import (
    estimate_homography,
    load_anchor_points_csv,
    to_id,
    transform_points,
)

## Handling Image Files

In [27]:
# Update these paths
anchors_dir = Path("/Volumes/LUCY DISK/mia/Project 1/example test/anchor_images")
tests_dir = Path("/Volumes/LUCY DISK/mia/Project 1/example test/test_images")
anchor_points_dir = Path("/Volumes/LUCY DISK/mia/Project 1/example test/anchor_images") # same folder as anchor images

grouping_results_csv = Path("/Users/lucywu/mia-final-1/grouping_results.csv") # GENERATED from assign_anchors.ipynb
submission_dir = Path("/Users/lucywu/mia-final-1/submission_step2") # GENERATED

# Matching / geometry params
ratio_thresh = 0.75
ransac_thresh = 3.0
nfeatures = 4000

assert anchors_dir.exists(), f"Missing anchors_dir: {anchors_dir}"
assert tests_dir.exists(), f"Missing tests_dir: {tests_dir}"
assert grouping_results_csv.exists(), f"Missing grouping_results_csv: {grouping_results_csv}"
assert anchor_points_dir.exists(), f"Missing anchor_points_dir: {anchor_points_dir}"

submission_dir.mkdir(parents=True, exist_ok=True)
(submission_dir / "correspondence").mkdir(parents=True, exist_ok=True)
(submission_dir / "warped_coordinates").mkdir(parents=True, exist_ok=True)

print("submission_dir:", submission_dir)

submission_dir: /Users/lucywu/mia-final-1/submission_step2


In [28]:
## Shared registration utilities

In [29]:
grouping_df = pd.read_csv(grouping_results_csv)
required = {"test_image", "anchor_image"}
missing_cols = required - set(grouping_df.columns)
if missing_cols:
    raise ValueError(f"grouping_results.csv missing columns: {missing_cols}")

grouping_submission_rows: List[Dict[str, str]] = []

for _, row in grouping_df.iterrows():
    test_name = str(row["test_image"]).strip()
    anchor_name = str(row["anchor_image"]).strip()

    test_path = tests_dir / test_name
    anchor_path = anchors_dir / anchor_name

    if not test_path.exists():
        raise FileNotFoundError(f"Missing test image: {test_path}")
    if not anchor_path.exists():
        raise FileNotFoundError(f"Missing anchor image: {anchor_path}")

    anchor_pts_df = load_anchor_points_csv(anchor_name, anchor_points_dir)

    anchor_img_bgr = cv2.imread(str(anchor_path), cv2.IMREAD_COLOR)
    test_img_bgr = cv2.imread(str(test_path), cv2.IMREAD_COLOR)

    H, _stats = estimate_homography(
        anchor_img_bgr=anchor_img_bgr,
        test_img_bgr=test_img_bgr,
        ratio_thresh=ratio_thresh,
        ransac_thresh=ransac_thresh,
        nfeatures=nfeatures,
    )

    points_xy = anchor_pts_df[["x", "y"]].to_numpy(dtype=np.float32)
    warped_xy = transform_points(points_xy, H)

    test_id = to_id(test_name, "test")
    anchor_id = to_id(anchor_name, "anchor")

    warped_out = pd.DataFrame({"x": warped_xy[:, 0], "y": warped_xy[:, 1]})
    warped_csv_path = submission_dir / "warped_coordinates" / f"{test_id}_warped_coordinates.csv"
    warped_out.to_csv(warped_csv_path, index=False, float_format="%.2f")

    grouping_submission_rows.append({"test_id": test_id, "anchor_id": anchor_id})

grouping_submission_df = pd.DataFrame(grouping_submission_rows).sort_values("test_id")
grouping_submission_df.to_csv(submission_dir / "correspondence" / "grouping.csv", index=False)

print(f"Saved grouping file: {submission_dir / 'correspondence' / 'grouping.csv'}")
print(f"Saved warped coordinate files in: {submission_dir / 'warped_coordinates'}")
print("\nPreview:")
display(grouping_submission_df.head())


Saved grouping file: /Users/lucywu/mia-final-1/submission_step2/correspondence/grouping.csv
Saved warped coordinate files in: /Users/lucywu/mia-final-1/submission_step2/warped_coordinates

Preview:


,test_id,anchor_id
0,test_01,anchor_04
1,test_02,anchor_02
2,test_03,anchor_01
3,test_04,anchor_01
4,test_05,anchor_04


## Optional evaluation (use for testing)

In [30]:
import subprocess
import sys

gt_dir = Path('/Volumes/LUCY DISK/mia/Project 1/example test/ground_truth')
subprocess.run([
    sys.executable,
    '/Users/lucywu/mia-final-1/eval.py',
    '--pred_dir', str(submission_dir),
    '--gt_dir', str(gt_dir),
    '--skip', 'segmentation',
    '-v',
], check=True)

  MIA 2026 Project 1 — Evaluation Results

  Task 1: Anchor Assignment (Grouping)
------------------------------------------------------------------------
  Accuracy:  0.8000  (20/25)

  Incorrect assignments (5):
    test_07:  predicted=anchor_04,  correct=anchor_05
    test_08:  predicted=anchor_04,  correct=anchor_05
    test_16:  predicted=anchor_04,  correct=anchor_05
    test_19:  predicted=anchor_04,  correct=anchor_05
    test_22:  predicted=anchor_05,  correct=anchor_04

  Task 2: Warped Coordinate Prediction (Registration)
------------------------------------------------------------------------
  MSE:       246189.0453
  Evaluated: 25/50 test images

  Per-image MSE:
    test_01:  MSE = 5923.2691
    test_02:  MSE = 8073.3958
    test_03:  MSE = 10737.8083
    test_04:  MSE = 1089.9268
    test_05:  MSE = 3762.8405
    test_06:  MSE = 9455.7426
    test_07:  MSE = 1307153.1459
    test_08:  MSE = 1159456.0340
    test_09:  MSE = 25432.1768
    test_10:  MSE = 5683.7000
    te

CompletedProcess(args=['/Users/lucywu/quantum/miniconda3/envs/mia-final-1/bin/python', '/Users/lucywu/mia-final-1/eval.py', '--pred_dir', '/Users/lucywu/mia-final-1/submission_step2', '--gt_dir', '/Volumes/LUCY DISK/mia/Project 1/example test/ground_truth', '--skip', 'segmentation', '-v'], returncode=0)